In [2]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import db_dtypes
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [3]:
sql = f"""
WITH payment_sessions AS (
  SELECT
    DeviceId,
    SessionId,
    MIN(Time) AS first_payment_time
  FROM `openrice-production.ORGA.PV_20260915`
  WHERE DeviceId IS NOT NULL
    AND SessionId IS NOT NULL
    AND (
      LOWER(EventAction) LIKE '%takeaway.pay%'
      OR LOWER(EventLabelRaw) LIKE '%takeaway.pay%'
    )
  GROUP BY DeviceId, SessionId
  ORDER BY first_payment_time
  LIMIT 20
)

SELECT pv.*
FROM `openrice-production.ORGA.PV_20260915` AS pv
INNER JOIN payment_sessions AS payment
  ON pv.DeviceId = payment.DeviceId
  AND pv.SessionId = payment.SessionId
ORDER BY payment.first_payment_time, pv.DeviceId, pv.SessionId, pv.Time;
"""

#Execute query
df_bq = client.query(sql).result().to_dataframe()
df_bq

C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,SessionId,DeviceId,Time,UserId,IP,EventAction,EventCategory,EventData,EventEntity,EventLabel,EventLabelRaw,EventSource,UserAgent,Product,MachineName,CollectTime,Platform
0,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,2026-09-15 00:43:59.500000+00:00,ee1fd8bc-9843-44d2-95d2-b0fab193b1eb,123.202.105.130,or.app.resume,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 00:23:01+00:00,ios
1,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,2026-09-15 00:44:00+00:00,ee1fd8bc-9843-44d2-95d2-b0fab193b1eb,123.202.105.130,or.app.start,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...","CityID:0;Lang:hk;Ver:7.20.5;device:iPhone18,1",Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 00:23:01+00:00,ios
2,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,2026-09-15 00:44:06.700000+00:00,ee1fd8bc-9843-44d2-95d2-b0fab193b1eb,123.202.105.130,or.qcksearch,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;sr:Index;sn:HK.Home,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 00:23:01+00:00,ios
3,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,2026-09-15 00:44:08.800000+00:00,ee1fd8bc-9843-44d2-95d2-b0fab193b1eb,123.202.105.130,or.search.quick,,,,"{'list': [{'item': {'List': 0, 'Param': 'Page'...",Page:1;whatwhere:串燒兄妹;savehistory:true;CityID:...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 00:23:01+00:00,ios
4,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,2026-09-15 00:44:09.100000+00:00,ee1fd8bc-9843-44d2-95d2-b0fab193b1eb,123.202.105.130,impression.poi,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;POIID:769806;Type:...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 00:23:01+00:00,ios
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
388,311131,4d5b5e4a-c1e3-4fb0-8b3e-8523d6398915,2026-09-15 07:54:57.400000+00:00,cc0367a7-34c5-4a73-8058-e8fca4deabdd,14.0.152.96,or.takeaway.order,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;POIID:697043;sr:me...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 07:34:01+00:00,ios
389,311131,4d5b5e4a-c1e3-4fb0-8b3e-8523d6398915,2026-09-15 07:55:05.400000+00:00,cc0367a7-34c5-4a73-8058-e8fca4deabdd,14.0.152.96,or.takeaway.checkout,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;sr:menu;POIID:6970...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 07:34:01+00:00,ios
390,311131,4d5b5e4a-c1e3-4fb0-8b3e-8523d6398915,2026-09-15 07:55:22.400000+00:00,cc0367a7-34c5-4a73-8058-e8fca4deabdd,14.0.152.96,or.takeaway.place-order,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;sr:checkout;POIID:...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 07:34:01+00:00,ios
391,311131,4d5b5e4a-c1e3-4fb0-8b3e-8523d6398915,2026-09-15 07:55:27.200000+00:00,cc0367a7-34c5-4a73-8058-e8fca4deabdd,14.0.152.96,or.takeaway.pay,,,,"{'list': [{'item': {'List': 0, 'Param': 'CityI...",CityID:0;Lang:hk;Ver:7.20.5;sr:payment;poiId:6...,Events,OpenRice_iOS/7.20.5 (iPhone; iOS 26.6.2);Lang:...,OpenRice,production-orga-openrice-netcore-app-67b5c6675...,2026-09-15 07:34:01+00:00,ios


In [4]:
#df_bq.to_csv("output.csv", index=False, encoding="utf-8-sig")


In [5]:
import pandas as pd
import json

# 將 EventAction 對應成宏觀行為標籤
def classify_event(event_action):
    action = str(event_action).strip().lower()

    event_mapping = {
        "or.app.start": "開啟 App",
        "or.app.resume": "返回 App",
        "or.qcksearch": "開啟快速搜尋",
        "or.search.quick": "搜尋餐廳",
        "or.search.layer": "搜尋頁面",
        "or.search.layer.record": "使用搜尋紀錄",
        "or.search.get-poi": "從搜尋結果進入 POI",
        "impression.poi": "瀏覽POI",
        "or.poi.get-details": "載入 POI 詳情",
        "or.poi.get-overview": "載入 POI 概覽",
        "or.poi.back": "離開 POI 頁面",
        "or.takeaway.order": "進入外賣自取點餐頁",
        "poi.emenu.get-item": "瀏覽菜單",
        "or.takeaway.checkout": "前往結帳",
        "or.takeaway.place-order": "確認下單",
        "or.takeaway.pay": "付款",
        "user.bookmark.poi": "新增收藏",
        "or.myor.bkmpoi": "從收藏清單進入 POI",
        "myor.search.bkmpoi": "瀏覽收藏清單",
        "or.search.themelist.takeaway": "點擊外賣自取按鈕",

        "or.myor.orderlist": "載入「我的訂單」列表",
        "or.poi.get-photos.food": "載入餐廳食物相片",
        "or.search.layer.tips": "使用搜尋提示",
        "or.poi.get-photos.menu": "載入餐廳菜單相片",
        "or.app.openpush": "從 push notification 開啟 App",
        "myor.search.bkmpoi": "從收藏清單搜尋收藏 POI ",
        "user.review.write": "撰寫評論",
        "user.myor": "進入 MyOR/個人中心",
        "or.poi.get-reviews": "載入餐廳評論",
        "or.poi.review.back": "離開餐廳評論",
        "or.search.nearby": "附近餐廳搜尋",
       "impression.sponsor.poi": "瀏覽贊助餐廳",
        "or.poi.map": "查看餐廳地圖",
        "or.myor.bkmpoi": "從收藏清單選擇餐廳",
        "or.takeaway.view-basket": "查看外賣購物籃",
        "or.deeplink.get-poi": "由 deep link 獲取 POI",
        "view.sr1.promotion": "查看搜尋結果中的廣告",
        "or.poi.photo.detail": "查看餐廳相片詳情",
        "user.share.poi": "分享餐廳"
    }

    return event_mapping.get(action, "其他事件")

# 時間排序，避免同一 session 的 event 順序混亂
tracking_source = df_bq.copy()
tracking_source["Time"] = pd.to_datetime(tracking_source["Time"])
tracking_source = tracking_source.sort_values(
    ["DeviceId", "SessionId", "Time"]
)

# 保留 EventAction 原文，並產生對應的行為判斷
tracking_source["event_action_raw"] = (
    tracking_source["EventAction"]
    .fillna("(empty event action)")
    .astype(str)
)

tracking_source["event_journey"] = (
    tracking_source["EventAction"]
    .apply(classify_event)
)

# 每個 DeviceId + SessionId 一行；timeline 與 journey 的相同 index 互相對應
behavior_tracking = (
    tracking_source
    .groupby(["SessionId", "DeviceId"], as_index=False)
    .agg(
        event_timeline=(
            "event_action_raw",
            lambda events: json.dumps(
                events.tolist(),
                ensure_ascii=False,
                indent=2
            )
        ),
        journey=(
            "event_journey",
            lambda indicators: json.dumps(
                indicators.tolist(),
                ensure_ascii=False,
                indent=2
            )
        ),
    )
    .rename(
        columns={
            "SessionId": "session_id",
            "DeviceId": "device_id",
        }
    )
)

behavior_tracking



,session_id,device_id,event_timeline,journey
0,10650,029c3705953a0a62,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""開啟 App"",\n ""開..."
1,119842,1daf763b-8da1-4b2f-b824-12d558a8ffda,"[\n ""or.app.resume"",\n ""impression.poi"",\n ...","[\n ""返回 App"",\n ""瀏覽POI"",\n ""瀏覽POI"",\n ""瀏覽P..."
2,123448,1ea45fc0-b9e7-4835-8ebe-223344650e0a,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""返回 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜..."
3,127050,1f894cab868b116f,"[\n ""or.poi.get-overview"",\n ""or.poi.get-det...","[\n ""載入 POI 概覽"",\n ""載入 POI 詳情"",\n ""載入 POI 概..."
4,178112,2c1ec027-be02-4201-a8ce-09c4ca6031b2,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""返回 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜..."
5,240829,3bc6f903023545c0,"[\n ""or.app.start"",\n ""or.app.start"",\n ""or...","[\n ""開啟 App"",\n ""開啟 App"",\n ""點擊外賣自取按鈕"",\n ..."
6,241938,3c09a1ff-e829-4900-a029-8fb5d8bcc6bf,"[\n ""or.app.start"",\n ""or.qcksearch"",\n ""or...","[\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜尋頁面"",\n ""搜尋餐..."
7,252253,3ea40686-e25e-4ccb-9887-64c71db37c78,"[\n ""or.app.resume"",\n ""or.qcksearch"",\n ""o...","[\n ""返回 App"",\n ""開啟快速搜尋"",\n ""載入 POI 詳情"",\n ..."
8,275540,447732ed-a9da-4a73-8f5b-e8a6fd65ed1e,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""o...","[\n ""返回 App"",\n ""開啟 App"",\n ""開啟快速搜尋"",\n ""搜..."
9,311131,4d5b5e4a-c1e3-4fb0-8b3e-8523d6398915,"[\n ""or.app.resume"",\n ""or.app.start"",\n ""u...","[\n ""返回 App"",\n ""開啟 App"",\n ""進入 MyOR/個人中心"",..."


In [6]:
behavior_tracking.to_csv("behavior_tracking.csv", index=False, encoding="utf-8-sig")

In [7]:
def list_unknown_events(dataframe):
    event_mapping = {
        "or.app.start",
        "or.app.resume",
        "or.qcksearch",
        "or.search.quick",
        "or.search.layer",
        "or.search.layer.record",
        "or.search.get-poi",
        "impression.poi",
        "or.poi.get-details",
        "or.poi.get-overview",
        "or.poi.back",
        "or.takeaway.order",
        "poi.emenu.get-item",
        "or.takeaway.checkout",
        "or.takeaway.place-order",
        "or.takeaway.pay",
        "user.bookmark.poi",
        "or.myor.bkmPOI",
        "myor.search.bkmPOI",
        "or.search.themelist.takeaway",

        "or.myor.orderlist",
        "or.poi.get-photos.food",
        "or.search.layer.tips",
        "or.poi.get-photos.menu",
        "or.app.openpush",
        "myor.search.bkmpoi",
        "user.review.write",
        "user.myor",
        "or.poi.get-reviews",
        "or.poi.review.back",
        "or.search.nearby",
        "impression.sponsor.poi",
        "or.poi.map",
        "or.myor.bkmpoi",
        "or.takeaway.view-basket",
        "or.deeplink.get-poi",
        "view.sr1.promotion",
        "or.poi.photo.detail",
        "user.share.poi"
    }

    normalized_actions = (
        dataframe["EventAction"]
        .fillna("(empty event action)")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    unknown_events = (
        normalized_actions[~normalized_actions.isin(event_mapping)]
        .value_counts()
        .rename_axis("unknown_event_action")
        .reset_index(name="event_count")
    )

    return unknown_events


unknown_events = list_unknown_events(df_bq)
unknown_events

,unknown_event_action,event_count


In [8]:
unknown_event_details = tracking_source[
    tracking_source["EventAction"]
    .fillna("(empty event action)")
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(unknown_events["unknown_event_action"])
][
    ["SessionId", "DeviceId", "Time", "EventAction", "EventLabelRaw"]
]

unknown_event_details

,SessionId,DeviceId,Time,EventAction,EventLabelRaw
